<a href="https://colab.research.google.com/github/1daniel3333/genAI_HW/blob/main/Gemma_Tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Important Setup Notes / 重要設定注意事項:

To run this notebook successfully, please ensure you have the following tokens configured:
為確保此筆記本能順利運行，請確認您已配置以下憑證（Tokens）：

1.  **Hugging Face Token (HF_TOKEN) / Hugging Face 憑證 (HF_TOKEN):**
    *   Visit [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) to generate a new token (make sure it has 'read' access, at minimum).
    *    請前往 [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) 生成一個新的憑證（至少需要 'read' 權限）。
    *   Add this token to Colab secrets. Click the '🔑' icon in the left sidebar, create a new secret named `HF_TOKEN`, and paste your Hugging Face token as its value. Ensure 'Notebook access' is enabled.
    *    將此憑證新增到 Colab 的「秘密金鑰」中。點擊左側邊欄的「🔑」圖示，創建一個名為 `HF_TOKEN` 的新秘密金鑰，並將您的 Hugging Face 憑證貼上作為其值。請確保已啟用「筆記本存取權」。

2.  **Ngrok Auth Token (NGROK_TOKEN) / Ngrok 授權憑證 (NGROK_TOKEN):**
    *   Register or log in at [ngrok.com](https://ngrok.com/). Find your Authtoken on the dashboard. Copy it.
    *    請在 [ngrok.com](https://ngrok.com/) 註冊或登入。在儀表板上找到您的認證憑證 (Authtoken)。複製它。
    *   Add this token to Colab secrets. Click the '🔑' icon in the left sidebar, create a new secret named `NGROK_TOKEN`, and paste your ngrok authtoken as its value. Ensure 'Notebook access' is enabled.
    *    將此憑證新增到 Colab 的「秘密金鑰」中。點擊左側邊欄的「🔑」圖示，創建一個名為 `NGROK_TOKEN` 的新秘密金鑰，並將您的 ngrok 認證憑證貼上作為其值。請確保已啟用「筆記本存取權」。

**After configuring these secrets, you must restart the Colab runtime (Runtime -> Restart runtime) and then run all cells from the beginning to apply the changes.**

**配置完這些秘密金鑰後，您必須重新啟動 Colab 運行環境 (執行階段 -> 重新啟動執行階段)，然後從頭開始運行所有儲存格，以使更改生效。**

In [ ]:
!pip install accelerate bitsandbytes
!pip install fastapi uvicorn pyngrok transformers torch

## Google Drive Storage Reminder / Google Drive 儲存提醒:

This notebook will download and save model files to your Google Drive (`/content/drive/MyDrive/gemma-2-2b-it`). This allows for faster loading on subsequent runs, avoiding re-downloads. The first time you run this, Colab will pop up a window asking for authorization to access your Google Drive; please allow it to proceed. This process might take some time, depending on your internet speed and the model size.

本筆記本會將模型檔案下載並儲存到您的 Google Drive 中 (`/content/drive/MyDrive/gemma-2-2b-it`)，以便下次使用時可以快速載入，避免重複下載。首次運行時，Colab 會彈出一個視窗，要求您授權此筆記本存取您的 Google Drive，請允許以繼續。這個過程可能需要一些時間，具體取決於網路速度和模型大小。

In [ ]:
import threading
from fastapi import FastAPI, Response
import json
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn
import nest_asyncio
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from google.colab import userdata
from google.colab import drive # Import drive
import os
import shutil # Import shutil for directory removal
from typing import List
import asyncio

# Mount Google Drive
drive.mount('/content/drive')

# Kill any existing ngrok tunnels
ngrok.kill()

# 允許 FastAPI 在 Jupyter 這種非同步環境中執行
nest_asyncio.apply()

app = FastAPI()

# Retrieve Hugging Face token from Colab secrets using userdata
hf_token = userdata.get('HF_TOKEN')

model_id = "google/gemma-2-2b-it"

# Define the path to save/load the model in Google Drive
drive_model_path = "/content/drive/MyDrive/gemma-2-2b-it"

# Configure 4-bit quantization (Only apply if needed, removed for Gemma-2B as it uses bfloat16)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

# Check if the model directory exists in Google Drive
if os.path.exists(drive_model_path):
    print("Attempting to load model from Google Drive...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(drive_model_path)
        model = AutoModelForCausalLM.from_pretrained(
            drive_model_path,
            device_map="auto",
            torch_dtype=torch.bfloat16, # Use bfloat16 for Gemma models
            low_cpu_mem_usage=True,
        )
        print("Model loaded successfully from Google Drive.")
    except Exception as e:
        print(f"Error loading model from {drive_model_path}: {e}")
        print("Model in Google Drive appears corrupted or incomplete. Removing and re-downloading...")
        if os.path.exists(drive_model_path):
            shutil.rmtree(drive_model_path)
        print("Downloading and saving model to Google Drive...")
        tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.bfloat16, # Use bfloat16 for Gemma models
            low_cpu_mem_usage=True,
            token=hf_token
        )
        # Save the model to Google Drive
        tokenizer.save_pretrained(drive_model_path)
        model.save_pretrained(drive_model_path)
        print("Model downloaded and saved to Google Drive.")
else:
    print("Model not found in Google Drive. Downloading and saving...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16, # Use bfloat16 for Gemma models
        low_cpu_mem_usage=True,
        token=hf_token
    )
    # Save the model to Google Drive
    tokenizer.save_pretrained(drive_model_path)
    model.save_pretrained(drive_model_path)
    print("Model downloaded and saved to Google Drive.")

print("Model loading process completed.")

In [ ]:

class ChatRequest(BaseModel):
    prompt: str

# Function to encapsulate model generation logic
def your_gemma_generate(input_text: str) -> str:
    input_ids = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**input_ids, max_new_tokens=1024)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # The response from the model will include the original prompt. We want to extract only the reply.
    if response.startswith(input_text):
        gemma_reply = response[len(input_text):].strip()
    else:
        gemma_reply = response.strip()
    return gemma_reply

@app.get("/v1/models")
def list_models():
    return {
        "object": "list",
        "data": [{
            "id": "gemma-2-2b-it",
            "object": "model",
            "owned_by": "google"
        }]
    }

@app.post("/v1/chat")
def chat(request: ChatRequest):
    gemma_reply = your_gemma_generate(request.prompt)
    return {"reply": f"【Gemma 回應】{gemma_reply}"}

class Message(BaseModel):
    role: str
    content: str

class CompletionRequest(BaseModel):
    model: str = "gemma-2-2b-it" # Updated default model name
    messages: List[Message]
    stream: bool = False

from fastapi import Response

@app.post("/v1/chat/completions")
async def openai_compatible_chat(request: CompletionRequest):
    prompt = request.messages[-1].content
    loop = asyncio.get_event_loop()
    reply = await loop.run_in_executor(None, your_gemma_generate, prompt)

    if request.stream:
        # 整包 SSE 一次回傳，不用 StreamingResponse
        sse  = f"data: {json.dumps({'id':'chatcmpl-gemma','object':'chat.completion.chunk','model':request.model,'choices':[{'index':0,'delta':{'role':'assistant','content':''},'finish_reason':None}]})}\n\n"
        sse += f"data: {json.dumps({'id':'chatcmpl-gemma','object':'chat.completion.chunk','model':request.model,'choices':[{'index':0,'delta':{'content':reply},'finish_reason':None}]})}\n\n"
        sse += f"data: {json.dumps({'id':'chatcmpl-gemma','object':'chat.completion.chunk','model':request.model,'choices':[{'index':0,'delta':{},'finish_reason':'stop'}]})}\n\n"
        sse += "data: [DONE]\n\n"
        return Response(content=sse, media_type="text/event-stream")

    return {
        "id": "chatcmpl-gemma",
        "object": "chat.completion",
        "model": request.model,
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": reply},
            "finish_reason": "stop"
        }]
    }

In [ ]:
# 設定 ngrok 內網穿透 (需要先去 ngrok 官網註冊拿免費的 Auth Token)
NGROK_TOKEN = os.environ.get("NGROK_TOKEN") # Retrieve NGROK_TOKEN from Colab secrets
ngrok.set_auth_token(NGROK_TOKEN)

# 開放 8000 連接埠
public_url = ngrok.connect(8000)
print(f"★ 你的 AI API 伺服器已上線！網址為: {public_url.public_url}")

# 在背景執行 Uvicorn 伺服器，避免卡死 Notebook
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server).start()